# Image Classification using CNN

Bu notebook **mushuk va itlarni** Convolutional Neural Network (CNN) yordamida tasniflaydi.

**Pipeline:**
1. Dataset yuklash (Kaggle'dan)
2. Data tayyorlash va preprocessing
3. CNN modelini qurish
4. Training
5. Natijalarni vizualizatsiya qilish
6. Test rasmlarda prediction

## 1. Muhitni sozlash

In [ ]:
# Google Colab'da GPU tekshirish
import tensorflow as tf
print('TensorFlow version:', tf.__version__)
print('GPU mavjud:', tf.config.list_physical_devices('GPU'))

# Agar GPU yo'q bo'lsa:
# Runtime -> Change runtime type -> Hardware accelerator -> GPU (T4)

In [ ]:
# Loyihani clone qilish
import os

REPO_URL = 'https://github.com/ozodbek-bosimov/image-classification-using-cnn.git'  # <- O'z repo URL'ingizni yozing
PROJECT_DIR = '/content/image-classification-using-cnn'

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
    print('Loyiha clone qilindi')
else:
    !cd {PROJECT_DIR} && git pull
    print('Loyiha yangilandi')

## 2. Dataset yuklash

Ushbu qadamda loyiha uchun kerakli ma'lumotlarni to'g'ridan-to'g'ri Google Drive'dan yuklab olamiz.

In [ ]:
# Dataset yuklash
import os
import zipfile

FILE_ID = '19inwa0n1W4DZamjCOm5XAlztvqG_xkjP'
LOCAL_ZIP_PATH = f'{PROJECT_DIR}/dataset.zip'

os.makedirs(PROJECT_DIR, exist_ok=True)

if not os.path.exists(f'{PROJECT_DIR}/train'):
    print('Dataset yuklanmoqda...')
    !gdown {FILE_ID} -O {LOCAL_ZIP_PATH} --quiet

    print('Asosiy ZIP fayldan chiqarilmoqda...')
    with zipfile.ZipFile(LOCAL_ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(PROJECT_DIR)
    os.remove(LOCAL_ZIP_PATH)

    print('Ichki ZIP fayllar (train/test) ochilmoqda...')
    for zip_name in ['train.zip', 'test1.zip']:
        inner_zip = os.path.join(PROJECT_DIR, zip_name)
        if os.path.exists(inner_zip):
            with zipfile.ZipFile(inner_zip, 'r') as zip_ref:
                zip_ref.extractall(PROJECT_DIR)
            os.remove(inner_zip)

    print('Dataset tayyor!')
else:
    print('Dataset allaqachon tayyor!')

# Tekshirish
try:
    train_count = len(os.listdir(f'{PROJECT_DIR}/train'))
    test_count = len(os.listdir(f'{PROJECT_DIR}/test1'))
    print(f'Train rasmlar soni: {train_count}')
    print(f'Test rasmlar soni: {test_count}')
except FileNotFoundError:
    print('Muammo yuz berdi, papkalar topilmadi.')

## 3. Loyiha kodini import qilish

In [ ]:
import sys
sys.path.insert(0, f'{PROJECT_DIR}/Code')

import constants as CONST
from data_prep import prep_and_load_data, get_size_statistics
from model import get_model

import numpy as np
import cv2
import copy
import pickle
import matplotlib.pyplot as plt
from tensorflow.keras.callbacks import TensorBoard, EarlyStopping

# Colab uchun yo'llarni yangilash
CONST.TRAIN_DIR = f'{PROJECT_DIR}/train'
CONST.TEST_DIR = f'{PROJECT_DIR}/test1'

print(f'Train dir: {CONST.TRAIN_DIR}')
print(f'Test dir:  {CONST.TEST_DIR}')
print(f'IMG_SIZE:  {CONST.IMG_SIZE}')
print(f'DATA_SIZE: {CONST.DATA_SIZE}')

## 4. Data exploration

In [ ]:
# Rasm o'lchamlari statistikasi
get_size_statistics()

In [ ]:
# Namunaviy rasmlarni ko'rsatish
import random

sample_images = random.sample(os.listdir(CONST.TRAIN_DIR), 10)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for idx, img_name in enumerate(sample_images):
    img_path = os.path.join(CONST.TRAIN_DIR, img_name)
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    label = 'Cat' if img_name.startswith('cat') else 'Dog'
    
    ax = axes[idx // 5, idx % 5]
    ax.imshow(img)
    ax.set_title(f'{label} ({img.shape[0]}x{img.shape[1]})')
    ax.axis('off')

plt.suptitle('Namunaviy rasmlar', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Data tayyorlash

In [ ]:
%%time
# Rasmlarni yuklash va preprocessing
data = prep_and_load_data()

In [ ]:
# Train / Validation split
train_size = int(len(data) * CONST.SPLIT_RATIO)

train_data = data[:train_size]
test_data = data[train_size:]

train_images = np.array([i[0] for i in train_data], dtype=np.float32)
train_labels = np.array([i[1] for i in train_data], dtype=np.float32)

test_images = np.array([i[0] for i in test_data], dtype=np.float32)
test_labels = np.array([i[1] for i in test_data], dtype=np.float32)

print(f'Train images: {train_images.shape}')
print(f'Train labels: {train_labels.shape}')
print(f'Test images:  {test_images.shape}')
print(f'Test labels:  {test_labels.shape}')

# RAM tozalash
del data, train_data, test_data

# Sinf taqsimotini tekshirish
cat_count = int(train_labels[:, 0].sum())
dog_count = int(train_labels[:, 1].sum())
print(f'\nCats: {cat_count}, Dogs: {dog_count}')

## 6. Model qurish

In [ ]:
model = get_model()
model.summary()

## 7. Training

In [ ]:
from google.colab import drive
import os
import pickle
from tensorflow.keras.models import load_model
import time

# Google Drive ni ulaymiz
drive.mount('/content/drive')
save_dir = '/content/drive/MyDrive/ML_Models/cats_vs_dogs'
model_path = f'{save_dir}/cats_vs_dogs_cnn.h5'
history_path = f'{save_dir}/training_history.pkl'

if os.path.exists(model_path):
    print(f'Google Drive dan tayyor model yuklanmoqda... ({model_path})')
    model = load_model(model_path)
    print('Model yuklandi! Training o\'tkazib yuboriladi.')
    
    class DummyHistory:
        history = None
    history = DummyHistory()
    if os.path.exists(history_path):
        with open(history_path, 'rb') as f:
            history.history = pickle.load(f)
            print('Oldingi training grafigi ham Drive dan oqildi.')
else:
    print('Google Drive da model topilmadi. Noldan o\'qitish boshlanmoqda...')
    
    log_dir = f'{PROJECT_DIR}/logs/{int(time.time())}'
    tensorboard_cb = TensorBoard(log_dir=log_dir)

    early_stop_cb = EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True
    )

    EPOCHS = 15
    BATCH_SIZE = 50

    print(f'Training boshlandi... ({EPOCHS} epochs, batch_size={BATCH_SIZE})')
    print(f'Train: {len(train_images)} images, Val: {len(test_images)} images')
    print('-' * 60)

    history = model.fit(
        train_images, train_labels,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        verbose=1,
        validation_data=(test_images, test_labels),
        callbacks=[tensorboard_cb, early_stop_cb]
    )
    print('Training tugadi!')


## 8. Natijalarni vizualizatsiya qilish

In [ ]:
if history.history is not None:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Accuracy grafik
    ax1.plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
    ax1.plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
    ax1.set_title('Model Accuracy', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.legend(loc='lower right')
    ax1.grid(True, alpha=0.3)

    # Loss grafik
    ax2.plot(history.history['loss'], label='Train Loss', linewidth=2)
    ax2.plot(history.history['val_loss'], label='Val Loss', linewidth=2)
    ax2.set_title('Model Loss', fontsize=14, fontweight='bold')
    ax2.set_ylabel('Loss')
    ax2.set_xlabel('Epoch')
    ax2.legend(loc='upper right')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    os.makedirs(f'{PROJECT_DIR}/output', exist_ok=True)
    plt.savefig(f'{PROJECT_DIR}/output/training_results.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Yakuniy natijalar
    print(f"\nYakuniy natijalar:")
    print(f"   Train Accuracy: {history.history['accuracy'][-1]:.4f}")
    print(f"   Val Accuracy:   {history.history['val_accuracy'][-1]:.4f}")
    print(f"   Train Loss:     {history.history['loss'][-1]:.4f}")
    print(f"   Val Loss:       {history.history['val_loss'][-1]:.4f}")
else:
    print('Google Drive dan tayyor model yuklangani sababli epoxma-epox o\'qish grafikalari yo\'q.')
    print('Model tayyor holatda! Keling, uni Test datada tekshirib ko\'ramiz: ')
    loss, acc = model.evaluate(test_images, test_labels, verbose=0)
    print(f"-> Test ma'lumotlaridagi aniqlik (Accuracy): {acc*100:.2f}%")
    print(f"-> Test loss: {loss:.4f}")


## 9. Modelni saqlash

In [ ]:
# Modelni saqlash
model_save_path = f'{PROJECT_DIR}/Code/models'
os.makedirs(model_save_path, exist_ok=True)

model.save(f'{model_save_path}/cats_vs_dogs_cnn.h5')
print(f'Model saqlandi: {model_save_path}/cats_vs_dogs_cnn.h5')

# History saqlash
with open(f'{model_save_path}/training_history.pickle', 'wb') as f:
    pickle.dump(history.history, f)
print(f'Training history saqlandi')

## 10. Test rasmlarda prediction

In [ ]:
def predict_and_show(model, image_paths, num_images=10):
    """Test rasmlarni predict qilib, natijalarni ko'rsatadi"""
    val_map = {0: 'Cat Cat', 1: 'Dog Dog'}
    
    cols = 5
    rows = (num_images + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(15, rows * 3))
    if rows == 1:
        axes = axes.reshape(1, -1)
    
    for idx in range(num_images):
        img_path = os.path.join(CONST.TEST_DIR, image_paths[idx])
        img = cv2.imread(img_path)
        if img is None:
            continue
        img_display = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Preprocessing
        img_resized = cv2.resize(img, (CONST.IMG_SIZE, CONST.IMG_SIZE))
        img_normalized = img_resized.astype('float32') / 255.0
        img_batch = img_normalized.reshape(1, CONST.IMG_SIZE, CONST.IMG_SIZE, 3)
        
        # Prediction
        pred = model.predict(img_batch, verbose=0)
        class_idx = np.argmax(pred[0])
        confidence = np.max(pred[0]) * 100
        
        ax = axes[idx // cols, idx % cols]
        ax.imshow(img_display)
        color = 'green' if confidence > 80 else 'orange'
        ax.set_title(f'{val_map[class_idx]}\n{confidence:.1f}%', 
                     fontsize=12, fontweight='bold', color=color)
        ax.axis('off')
    
    # Bo'sh kataklarni yashirish
    for idx in range(num_images, rows * cols):
        axes[idx // cols, idx % cols].axis('off')
    
    plt.suptitle('Test Predictions', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()


# Test rasmlardan namuna olish
test_images_list = os.listdir(CONST.TEST_DIR)
random.shuffle(test_images_list)
predict_and_show(model, test_images_list, num_images=10)

## 11. O'z rasminigzni tekshirish

Quyidagi cell orqali o'z rasmingizni yuklab tekshirishingiz mumkin.

In [ ]:
# O'z rasmingizni yuklang
from google.colab import files

uploaded = files.upload()

for filename in uploaded.keys():
    img = cv2.imread(filename)
    if img is None:
        print(f'{filename} o\'qib bo\'lmadi')
        continue
    
    img_display = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img, (CONST.IMG_SIZE, CONST.IMG_SIZE))
    img_normalized = img_resized.astype('float32') / 255.0
    img_batch = img_normalized.reshape(1, CONST.IMG_SIZE, CONST.IMG_SIZE, 3)
    
    pred = model.predict(img_batch, verbose=0)
    class_idx = np.argmax(pred[0])
    confidence = np.max(pred[0]) * 100
    
    val_map = {0: 'Cat Cat', 1: 'Dog Dog'}
    
    plt.figure(figsize=(6, 6))
    plt.imshow(img_display)
    plt.title(f'{val_map[class_idx]} — {confidence:.2f}%', fontsize=16, fontweight='bold')
    plt.axis('off')
    plt.show()
    
    print(f'\nNatija: {val_map[class_idx]}')
    print(f'Cat ehtimoli: {pred[0][0]*100:.2f}%')
    print(f'Dog ehtimoli: {pred[0][1]*100:.2f}%')

## 12. Modelni Google Drive'ga saqlash (ixtiyoriy)

Modelni doimiy saqlash uchun Google Drive'ga ko'chirish.

In [ ]:
# Modelni kelajak uchun Google Drive'ga saqlash
import os
import pickle

save_dir = '/content/drive/MyDrive/ML_Models/cats_vs_dogs'
os.makedirs(save_dir, exist_ok=True)

model.save(f'{save_dir}/cats_vs_dogs_cnn.h5')

# Tarix (grafik uchun) ma'lumotlarini ham saqlab qo'yamiz
if history.history is not None:
    with open(f'{save_dir}/training_history.pkl', 'wb') as f:
        pickle.dump(history.history, f)

print(f'Model va natijalar Google Drive\'ga saqlandi: {save_dir}')
